# Hardware test — `IviumsoftInstanceManager` (`feat/instance-manager`)

Exercises the instance lifecycle manager against real IviumSoft processes:

1. `launch()` — spawn + diff-detect the driver instance number (**timed** — feeds the default `launch_timeout`)
2. Two launches — each gets its own instance number and PID
3. `list_instances()` — managed records vs. pre-existing orphans
4. `close()` — graceful WM_CLOSE, instance disappears from the driver (**timed**)
5. Busy guard — refuses to close a measuring instance (manual step)
6. `adopt()` — re-attach to an instance launched outside the manager

**Prerequisites**
- Windows with IviumSoft installed, x64 Python, repo branch `feat/instance-manager`
- ⚠️ **Do not run while real measurements are in progress** — this notebook opens and closes IviumSoft instances.
- No devices needed except for step 5 (busy guard), which is optional and manual.

The escalation path (WM_CLOSE ignored → terminate) is deliberately **not** tested here — it requires wedging
a real instance behind a modal dialog. It is covered by the unit tests (`tests/test_instance_manager.py`).

In [ ]:
import sys
import time
from pathlib import Path

# Import the local dev copy (this branch), not a pip-installed pyvium.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists():
    if repo_root.parent == repo_root:
        raise RuntimeError("repo root not found — run this notebook from inside the pyvium repo")
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pyvium
from pyvium import IviumsoftInstanceManager, Pyvium
from pyvium.errors import DeviceBusyError
from pyvium.util import windows_process

print("pyvium imported from:", pyvium.__file__)

IVIUMSOFT_EXE = r"C:\IviumStat\IviumSoft.exe"  # adjust if needed
assert Path(IVIUMSOFT_EXE).exists(), f"IviumSoft not found at {IVIUMSOFT_EXE} — adjust the path"

### Open the driver and record the baseline
Instances already running before this notebook are *orphans* from the manager's point of view
(it knows their instance number but not their PID).

In [ ]:
Pyvium.open_driver()
baseline = set(Pyvium.get_active_iviumsoft_instances())
print("Pre-existing instances (orphans):", sorted(baseline) or "none")

manager = IviumsoftInstanceManager(exe_path=IVIUMSOFT_EXE, launch_timeout=60.0)

## 1–2. `launch()` twice — timed
Each launch spawns IviumSoft, then polls the driver until a **new** instance number appears
(the before/after diff is what maps PID → instance number).
Note the registration time: it should comfortably fit the default `launch_timeout=30` —
if it doesn't on the customer VM, that default needs raising.

In [ ]:
start = time.perf_counter()
record_1 = manager.launch()
elapsed_1 = time.perf_counter() - start
print(f"launch 1: {record_1} — registered in {elapsed_1:.1f}s")

start = time.perf_counter()
record_2 = manager.launch()
elapsed_2 = time.perf_counter() - start
print(f"launch 2: {record_2} — registered in {elapsed_2:.1f}s")

assert record_1.instance_number not in baseline
assert record_2.instance_number not in baseline
assert record_1.instance_number != record_2.instance_number, "both launches claimed the same instance"
assert record_1.pid != record_2.pid
assert record_1.managed and record_2.managed
print("PASS — each launch got its own instance number and PID")

## 3. `list_instances()` — managed vs. orphans
Launched instances carry their PID (`managed=True`); pre-existing ones appear with `pid=None`.

In [ ]:
listed = {item.instance_number: item for item in manager.list_instances()}
for number in sorted(listed):
    print(listed[number])

assert listed[record_1.instance_number].managed
assert listed[record_2.instance_number].managed
for orphan_number in baseline:
    assert listed[orphan_number].pid is None
    assert not listed[orphan_number].managed
print("PASS — managed and orphan records are distinguished")

## 4. `close()` — graceful, timed
Sends WM_CLOSE natively (no PowerShell) and waits for the process to exit.
Afterwards the driver must no longer list the instance and the PID must be gone.

In [ ]:
closing_number, closing_pid = record_2.instance_number, record_2.pid

start = time.perf_counter()
manager.close(closing_number)
elapsed = time.perf_counter() - start
print(f"close: instance {closing_number} (pid {closing_pid}) in {elapsed:.1f}s")

time.sleep(2)  # give the driver a moment to settle its bookkeeping
active_after = Pyvium.get_active_iviumsoft_instances()
assert closing_number not in active_after, f"driver still lists instance {closing_number}: {active_after}"
assert not windows_process.is_process_running(closing_pid), "process still alive after close"
print("PASS — instance gone from the driver, process exited, no escalation")

## 5. Busy guard (optional, manual)
Needs a device: **connect a device in the remaining launched instance and start any method from
the IviumSoft GUI**, then run this cell. `close()` must refuse with `DeviceBusyError`.
The cell skips itself if the instance is not measuring. (It does not force-close — your
measurement survives.)

In [ ]:
status_code, status_label = Pyvium.device(record_1.instance_number).get_device_status()
print(f"instance {record_1.instance_number} status: {status_code} ({status_label})")

if status_code != 2:
    print("SKIPPED — instance not measuring; start a method in the GUI and re-run this cell")
else:
    try:
        manager.close(record_1.instance_number)
    except DeviceBusyError as error:
        print(f"PASS — close refused while measuring: {error}")
    else:
        raise AssertionError("FAIL — close() did not refuse a busy instance")

## 6. `adopt()` — restart recovery
Simulates the API-server-restart scenario: an instance launched *outside* the manager
(plain `subprocess`, as a crashed server's child would appear), then re-attached via
`adopt(instance_number, pid)` and closed through the manager.

In [ ]:
import subprocess

before = set(Pyvium.get_active_iviumsoft_instances())
external = subprocess.Popen(
    [IVIUMSOFT_EXE],
    stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    close_fds=True,
)
print(f"external IviumSoft launched, pid {external.pid} — waiting for driver registration")

deadline = time.monotonic() + 60
new_instances = set()
while time.monotonic() < deadline:
    new_instances = set(Pyvium.get_active_iviumsoft_instances()) - before
    if new_instances:
        break
    time.sleep(0.5)
assert new_instances, "external instance never registered"
external_number = min(new_instances)

adopted = manager.adopt(external_number, external.pid)
print("adopted:", adopted)
assert adopted.pid == external.pid
assert not adopted.managed  # known pid, but no Popen handle inside the manager

manager.close(external_number)
time.sleep(2)
assert external_number not in Pyvium.get_active_iviumsoft_instances()
assert not windows_process.is_process_running(external.pid)
print("PASS — adopted instance closed gracefully through the manager")

## Cleanup
Closes every instance this manager still tracks (abort any measurement first — or pass
`force=True` consciously), then the driver. Pre-existing orphan instances are left untouched.

In [ ]:
for item in manager.list_instances():
    if item.pid is not None:
        try:
            manager.close(item.instance_number)
            print(f"closed instance {item.instance_number} (pid {item.pid})")
        except DeviceBusyError:
            print(f"instance {item.instance_number} is measuring — left running "
                  "(abort the method and re-run, or close with force=True)")

Pyvium.close_driver()
print("driver closed")